In [1]:
import os
import csv
import time
import random
import datetime
import numpy as np
import tensorflow as tf
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler

from tensorflow.keras.losses import Huber

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from mealpy import PSO
from mealpy.utils.problem import Problem, IntegerVar, FloatVar, MixedSetVar

from tcn import TCN

import gc

gpus = tf.config.experimental.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)


I0000 00:00:1773047080.284151  532497 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773047080.316500  532497 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773047081.045315  532497 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
df_hourly = pd.read_csv('cleandata.csv')
df_hourly.index = pd.to_datetime(df_hourly['timestamp'])
df_hourly.drop(columns=['timestamp'], inplace=True)

df_hourly.index = pd.to_datetime(df_hourly.index)
df_hourly = df_hourly.sort_index()
df_hourly.index.name = "Datetime"

# Calendar features
df_hourly["hour"] = df_hourly.index.hour
df_hourly["day_of_week"] = df_hourly.index.dayofweek

# Cyclical encoding
df_hourly["hour_sin"] = np.sin(2 * np.pi * df_hourly["hour"] / 24)
df_hourly["hour_cos"] = np.cos(2 * np.pi * df_hourly["hour"] / 24)
df_hourly["day_sin"]  = np.sin(2 * np.pi * df_hourly["day_of_week"] / 7)
df_hourly["day_cos"]  = np.cos(2 * np.pi * df_hourly["day_of_week"] / 7)

data = df_hourly["demand_kWh"]
features = df_hourly[["hour_sin", "hour_cos", "day_sin", "day_cos"]]


train_mask = (df_hourly.index >= "2018-01-01") & (df_hourly.index < "2022-01-01")
val_mask   = (df_hourly.index >= "2021-01-01") & (df_hourly.index < "2022-01-01")
test_mask  = (df_hourly.index >= "2022-01-01") & (df_hourly.index < "2023-01-01")

train = data.loc[train_mask]
val   = data.loc[val_mask]
test  = data.loc[test_mask]

train_features = features.loc[train_mask]
val_features   = features.loc[val_mask]
test_features  = features.loc[test_mask]

y_scaler = StandardScaler()

y_train = y_scaler.fit_transform(train.values.reshape(-1,1))
y_val   = y_scaler.transform(val.values.reshape(-1,1))
y_test  = y_scaler.transform(test.values.reshape(-1,1))

# Sin/cos features are already bounded in [-1, 1], so keep as-is
X_train_feat = train_features.values
X_val_feat   = val_features.values
X_test_feat  = test_features.values

WINDOW_SIZE = 168 * 3
HORIZON = 168

def create_sliding_windows(y, X, window_size, horizon):
    X_seq, y_seq = [], []

    total_samples = len(y) - window_size - horizon + 1

    for start in tqdm(range(total_samples), desc="Building sliding windows"):
        end_window = start + window_size
        end_target = end_window + horizon

        y_window = y[start:end_window]                 # (window_size, 1)
        X_window = X[start:end_window]                 # (window_size, 4)

        window_features = np.hstack([y_window, X_window])   # (window_size, 5)
        target_window = y[end_window:end_target].reshape(-1)  # (horizon,)

        X_seq.append(window_features)
        y_seq.append(target_window)

    return np.array(X_seq), np.array(y_seq)

X_train, y_train_seq = create_sliding_windows(y_train, X_train_feat, WINDOW_SIZE, HORIZON)
X_val,   y_val_seq   = create_sliding_windows(y_val,   X_val_feat,   WINDOW_SIZE, HORIZON)
X_test,  y_test_seq  = create_sliding_windows(y_test,  X_test_feat,  WINDOW_SIZE, HORIZON)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train_seq.shape)
print("X_val shape  :", X_val.shape)
print("y_val shape  :", y_val_seq.shape)
print("X_test shape :", X_test.shape)
print("y_test shape :", y_test_seq.shape)


Building sliding windows: 100%|██████████| 8089/8089 [00:00<00:00, 327800.94it/s]

X_train shape: (34376, 504, 5)
y_train shape: (34376, 168)
X_val shape  : (8089, 504, 5)
y_val shape  : (8089, 168)
X_test shape : (8089, 504, 5)
y_test shape : (8089, 168)


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [4]:
opt_technique = "PSO"
method = "TCN"

log_dir = f"./logs_{opt_technique}_{method}"
os.makedirs(log_dir, exist_ok=True)

In [5]:

def get_dilations(mode):
    if mode == "short":
        return [1, 2, 4, 8, 16]
    elif mode == "medium":
        return [1, 2, 4, 8, 16, 32]
    elif mode == "long":
        return [1, 2, 4, 8, 16, 32, 64]
    else:
        raise ValueError(f"Unknown dilation mode: {mode}")
    
def get_activation(x):
    if x == "relu":
        return "relu"
    elif x == "leaky_relu":
        return LeakyReLU(alpha=0.01)


def build_tcn_model(params, input_shape, horizon):
    dilations = get_dilations(params["dilation_mode"])

    if params["loss_function"] == "mse":
        loss = "mse"
    elif params["loss_function"] == "huber":
        loss = Huber()

    model = Sequential([
        Input(shape=input_shape),

        TCN(
            nb_filters=int(params["nb_filters"]),
            kernel_size=int(params["kernel_size"]),
            dilations=dilations,
            dropout_rate=float(params["dropout_rate"]),
            return_sequences=False
        ),

        Dense(int(params["dense_units_1"]), get_activation(params["activation_type"])),
        Dropout(float(params["dense_dropout_1"])),

        Dense(int(params["dense_units_2"]), get_activation(params["activation_type"])),
        Dropout(float(params["dense_dropout_2"])),

        Dense(horizon)
    ])

    model.compile(
        optimizer=Adam(learning_rate=float(params["learning_rate"])),
        loss=loss,
        metrics=["mse","mae"]
    )
    return model

In [ ]:

def train_tcn_with_params(params, log_file):
    model = None
    try:
        set_seed(42)

        input_shape = (X_train.shape[1], X_train.shape[2])

        model = build_tcn_model(
            params=params,
            input_shape=input_shape,
            horizon=HORIZON
        )

        start_time = time.time()

        history = model.fit(
            X_train,
            y_train_seq,
            validation_data=(X_val, y_val_seq),
            epochs=int(params["epochs"]),
            batch_size=int(params["batch_size"]),
            shuffle=False,
            verbose=0
        )

        train_time = time.time() - start_time

        # Validation prediction
        y_val_pred_scaled = model.predict(X_val, verbose=0)

        n_samples = y_val_pred_scaled.shape[0]
        horizon = y_val_pred_scaled.shape[1]

        y_val_pred = y_scaler.inverse_transform(
            y_val_pred_scaled.reshape(-1, 1)
        ).reshape(n_samples, horizon)

        y_val_true = y_scaler.inverse_transform(
            y_val_seq.reshape(-1, 1)
        ).reshape(n_samples, horizon)

        rmse = np.sqrt(mean_squared_error(y_val_true.reshape(-1), y_val_pred.reshape(-1)))
        mae = mean_absolute_error(y_val_true.reshape(-1), y_val_pred.reshape(-1))
        r2 = r2_score(y_val_true.reshape(-1), y_val_pred.reshape(-1))

        with open(log_file, "a") as f:
            f.write(
                f"params={params}, "
                f"val_rmse={rmse:.6f}, "
                f"val_mae={mae:.6f}, "
                f"val_r2={r2:.6f}, "
                f"time={train_time:.2f}s\n"
            )

        return float(rmse)

    except Exception as e:
        print(f"Error occurred: {e}")
        return 9999.0

    finally:
        # MEMORY CLEANUP


        try:
            del model
        except:
            pass

        try:
            del history
        except:
            pass

        try:
            del y_val_pred_scaled
        except:
            pass

        # Clear TensorFlow session
        tf.keras.backend.clear_session()

        # Run Python garbage collector
        gc.collect()

In [7]:
class OptimizeTCNProb(Problem):
    def __init__(self, bounds=None, minmax="min", name="TCN_PSO", **kwargs):
        super().__init__(bounds, minmax, **kwargs)
        self.log_file = os.path.join(log_dir, "training_log_PSO_TCN.txt")

    def obj_func(self, x):
        decoded = self.decode_solution(x)
        fitness = train_tcn_with_params(decoded, self.log_file)
        # print("Decoded params:", decoded)
        # print("Validation RMSE:", fitness)
        return float(fitness)

In [8]:
bounds = [
    MixedSetVar(valid_sets=[32, 64, 128], name="nb_filters"),
    MixedSetVar(valid_sets=[2, 3, 5, 7], name="kernel_size"),

    FloatVar(lb=0.1, ub=0.5, name="dropout_rate"),

    MixedSetVar(valid_sets=[32, 64, 128, 256], name="dense_units_1"),
    MixedSetVar(valid_sets=[16, 32, 64, 128], name="dense_units_2"),

    FloatVar(lb=0.1, ub=0.5, name="dense_dropout_1"),
    FloatVar(lb=0.1, ub=0.5, name="dense_dropout_2"),

    MixedSetVar(valid_sets=["short", "medium", "long"], name="dilation_mode"),

    MixedSetVar(valid_sets=[1e-4, 3e-4, 5e-4, 1e-3], name="learning_rate"),
    MixedSetVar(valid_sets=[16, 32], name="batch_size"),


    MixedSetVar(valid_sets=["relu", "leaky_relu"], name="activation_type"),
    MixedSetVar(valid_sets=["mse", "huber"], name="loss_function"),

    MixedSetVar(valid_sets=[30], name="epochs"),
]

In [9]:
now = datetime.datetime.now()
date_time_str = now.strftime("%Hh_%d_%m_%Y")

problem = OptimizeTCNProb(
    bounds=bounds,
    minmax="min",
    log_to="file",
    log_file=os.path.join(log_dir, f"PSO_results_TCN_{date_time_str}.txt")
)

model_pso = PSO.OriginalPSO(
    epoch=50,
    pop_size=10,
    mode="single"
)

termination = {"max_early_stop": 8}

g_best = model_pso.solve(problem, termination=termination)

print("Best solution vector:", g_best.solution)
print("Best fitness:", g_best.target.fitness)

best_params = problem.decode_solution(g_best.solution)
print("Best decoded params:", best_params)

model_pso.history.save_global_objectives_chart(
    filename=os.path.join(log_dir, "global_objectives")
)
model_pso.history.save_local_objectives_chart(
    filename=os.path.join(log_dir, "local_objectives")
)
model_pso.history.save_global_best_fitness_chart(
    filename=os.path.join(log_dir, "global_best_fitness")
)
model_pso.history.save_local_best_fitness_chart(
    filename=os.path.join(log_dir, "local_best_fitness")
)
model_pso.history.save_runtime_chart(
    filename=os.path.join(log_dir, "runtime")
)
model_pso.history.save_exploration_exploitation_chart(
    filename=os.path.join(log_dir, "explore_exploit")
)
model_pso.history.save_diversity_chart(
    filename=os.path.join(log_dir, "diversity")
)

I0000 00:00:1773047082.455298  532497 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20803 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
I0000 00:00:1773047085.491859  532593 service.cc:153] XLA service 0x701ca005a840 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773047085.491874  532593 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4090, Compute Capability 8.9 (Driver: 12.8.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.19.0)
I0000 00:00:1773047085.565994  532593 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1773047086.177061  532593 cuda_dnn.cc:461] Loaded cuDNN version 91900
I0000 00:00:1773047086.408058  532593 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8884__.41
I0000 00:00:1773047095.599769  532593 device_comp

KeyboardInterrupt: 

In [ ]:
csv_filename = os.path.join(log_dir, "PSO_results.csv")

data_to_save = {
    "Generation": range(1, len(model_pso.history.list_global_best_fit) + 1),
    "Global_Best_Fitness": model_pso.history.list_global_best_fit,
    "Current_Best_Fitness": model_pso.history.list_current_best_fit,
    "Diversity": model_pso.history.list_diversity,
    "Exploration": model_pso.history.list_exploration,
    "Exploitation": model_pso.history.list_exploitation,
}

with open(csv_filename, mode="w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=data_to_save.keys())
    writer.writeheader()

    for i in range(len(model_pso.history.list_global_best_fit)):
        row = {
            "Generation": data_to_save["Generation"][i],
            "Global_Best_Fitness": data_to_save["Global_Best_Fitness"][i],
            "Current_Best_Fitness": data_to_save["Current_Best_Fitness"][i],
            "Diversity": data_to_save["Diversity"][i],
            "Exploration": data_to_save["Exploration"][i],
            "Exploitation": data_to_save["Exploitation"][i],
        }
        writer.writerow(row)

print(f"Results saved in {csv_filename}")